# Phase 2.3 — Cross-Dataset SSL Transfer (BYOL + MoCo)

Three protocols, both SSL methods, one notebook.

- **Protocol A — Zero-shot probe:** load a source-pretrained encoder, freeze it, train a logistic-regression probe on target labels. Answers *"does the SSL representation transfer without any target fine-tuning?"*
- **Protocol C — Joint pretraining:** run BYOL/MoCo SSL on the union of all datasets' unlabeled snapshots, then probe each target independently. Answers *"does adding target unlabeled traffic help?"*
- **Protocol B — Few-shot fine-tune:** initialise the target's TFT encoder from a source SSL checkpoint, then fine-tune on `frac` of target labels. Answers *"how many labels do we save?"*

Three normalisation protocols are supported:

- `self`   — target's own train-split mean/std (in-domain baseline)
- `source` — apply source's stats to target (hardest, most honest transfer)
- `global` — a single set of mean/std computed from the union of all train splits

## Cell 1 — Imports

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, json, math, pickle, contextlib, warnings, time, random, copy, shutil
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import DataLoader, Batch
from torch_geometric.nn import GATConv, GlobalAttention
from torch.cuda.amp import GradScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                             silhouette_score, confusion_matrix)
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark        = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
torch.set_float32_matmul_precision('high')
warnings.filterwarnings('ignore')

## Cell 2 — Config

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_CONFIG = {
    'gotham': 'Gotham_processed_features_10s/graphs',
    'nf_bot': 'NFBoTIoT_v3_10s/graphs',
    'nf_ton': 'NFToNIoT_v3_10s/graphs',
}
SSL_METHODS = ['byol', 'moco']

# ---- shared architecture (must match the SSL notebooks exactly) ----
HIDDEN_DIM     = 128
NUM_HEADS      = 4
NUM_LAYERS     = 3
PROJECTION_DIM = 128
PREDICTOR_HID  = 256

# ---- BYOL hyperparameters ----
BYOL_MOMENTUM_BASE  = 0.996
BYOL_MOMENTUM_FINAL = 1.0

# ---- MoCo hyperparameters ----
QUEUE_SIZE   = 4096
TEMPERATURE  = 0.07
MOCO_MOMENTUM = 0.999

# ---- joint SSL optimisation ----
BATCH_SIZE       = 128
GRAD_ACCUM_STEPS = 2
SSL_LR           = 3e-4
SSL_WD           = 1e-4
SSL_EPOCHS       = 60
SSL_EVAL_EVERY   = 5
SSL_EARLY_STOP   = 30
RECON_WEIGHT     = 0.05
RECON_MASK_RATIO = 0.30
USE_MASKED_RECON = True

# ---- graph caps ----
MAX_NODES = 128
MAX_EDGES = 20000

# ---- probe ----
PROBE_TRAIN_MAX = 4000
KNN_K           = 10
SEED            = 42

# ---- cross-dataset evaluation policy ----
# fix ONE attack threshold across all datasets so transfer numbers are comparable
CROSS_DATASET_ATTACK_THRESHOLD = 0.10

# ---- per-dataset SSL augment rates (same as SSL notebooks) ----
EDGE_DROPOUT_MAP = {'gotham': 0.10, 'nf_ton': 0.10, 'nf_bot': 0.05}
FEATURE_MASK_MAP = {'gotham': 0.10, 'nf_ton': 0.10, 'nf_bot': 0.05}
NODE_DROP_MAP    = {'gotham': 0.05, 'nf_ton': 0.05, 'nf_bot': 0.02}
SUBGRAPH_MAP     = {'gotham': 0.90, 'nf_ton': 0.90, 'nf_bot': 0.95}

# ---- for joint pretraining, use the mean of per-dataset rates ----
JOINT_EDGE_DROP  = float(np.mean(list(EDGE_DROPOUT_MAP.values())))
JOINT_FEAT_MASK  = float(np.mean(list(FEATURE_MASK_MAP.values())))
JOINT_NODE_DROP  = float(np.mean(list(NODE_DROP_MAP.values())))
JOINT_SUBGRAPH   = float(np.mean(list(SUBGRAPH_MAP.values())))

# ---- output dirs ----
PLOTS_DIR = 'plots/cross_dataset'
os.makedirs(PLOTS_DIR, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f"Device        : {DEVICE}")
print(f"Datasets      : {list(DATASET_CONFIG.keys())}")
print(f"SSL methods   : {SSL_METHODS}")
print(f"Joint SSL     : epochs={SSL_EPOCHS}  bs={BATCH_SIZE}  lr={SSL_LR}")
print(f"Joint aug     : edge={JOINT_EDGE_DROP:.3f}  feat={JOINT_FEAT_MASK:.3f}  "
      f"node={JOINT_NODE_DROP:.3f}  sub={JOINT_SUBGRAPH:.3f}")
print(f"Attack thresh : {CROSS_DATASET_ATTACK_THRESHOLD}  (fixed across datasets)")

Device        : cuda
Datasets      : ['gotham', 'nf_bot', 'nf_ton']
SSL methods   : ['byol', 'moco']
Joint SSL     : epochs=60  bs=128  lr=0.0003
Joint aug     : edge=0.083  feat=0.083  node=0.040  sub=0.917
Attack thresh : 0.1  (fixed across datasets)


## Cell 3 — Graph utilities (identical to SSL notebooks)

In [3]:
def cap_graph(g, max_nodes=MAX_NODES, max_edges=MAX_EDGES):
    g = g.clone(); num_nodes = g.num_nodes
    if g.edge_index.size(1) > max_edges:
        keep = torch.randperm(g.edge_index.size(1))[:max_edges]
        g.edge_index = g.edge_index[:, keep]
        if g.edge_attr is not None and g.edge_attr.size(0) > 0:
            if g.edge_attr.dim() > 2:
                g.edge_attr = g.edge_attr.view(-1, g.edge_attr.size(-1))
            g.edge_attr = g.edge_attr[keep]
    if num_nodes > max_nodes:
        deg = torch.zeros(num_nodes, dtype=torch.long)
        deg.scatter_add_(0, g.edge_index[0],
                         torch.ones(g.edge_index.size(1), dtype=torch.long))
        kn = torch.topk(deg, max_nodes).indices
        mask = torch.zeros(num_nodes, dtype=torch.bool); mask[kn] = True
        s, d = g.edge_index; keep_e = mask[s] & mask[d]
        g.edge_index = g.edge_index[:, keep_e]
        if g.edge_attr is not None and g.edge_attr.size(0) > 0:
            g.edge_attr = g.edge_attr[keep_e]
        remap = torch.zeros(num_nodes, dtype=torch.long)
        remap[mask] = torch.arange(max_nodes, dtype=torch.long)
        g.edge_index = remap[g.edge_index]; g.x = g.x[mask]
    return g


def dominant_family(g, attack_threshold=CROSS_DATASET_ATTACK_THRESHOLD,
                    blacklist=('Unknown', 'unknown', 'UNKNOWN')):
    names = getattr(g, 'edge_attack_names', None)
    if not names: return 'Benign'
    c = Counter(names); total = sum(c.values())
    atk = {k: v for k, v in c.items() if k != 'Benign' and k not in blacklist}
    if not atk: return 'Benign'
    if sum(atk.values()) / max(total, 1) < attack_threshold: return 'Benign'
    return max(atk, key=atk.get)


def size_aware_rate(num_edges, base_rate):
    if num_edges < 500:  return base_rate * 0.4
    if num_edges < 2000: return base_rate * 0.7
    return base_rate


def _rate_vec(edge_counts, base_rate):
    c = edge_counts.to(torch.float32)
    r = torch.full_like(c, base_rate)
    r = torch.where(c < 2000, c.new_full((), base_rate * 0.7), r)
    r = torch.where(c < 500,  c.new_full((), base_rate * 0.4), r)
    return r


@torch.no_grad()
def augment_batch(batch, edge_drop, feat_mask, node_drop, subgraph):
    num_edges = batch.edge_index.size(1)
    device    = batch.edge_index.device
    num_nodes = batch.x.size(0)

    if edge_drop > 0.0 and num_edges > 0:
        edge_graph  = batch.batch[batch.edge_index[0]]
        edge_counts = torch.bincount(edge_graph, minlength=batch.num_graphs)
        edge_rates  = _rate_vec(edge_counts, edge_drop)[edge_graph]
    else:
        edge_rates = None

    eff_feat = size_aware_rate(num_edges, feat_mask) if feat_mask > 0.0 else 0.0
    do_node  = (node_drop > 0.0) or (subgraph < 1.0)

    def _one(b):
        if edge_rates is not None and b.edge_index.size(1) > 0:
            keep = torch.rand(b.edge_index.size(1), device=device) >= edge_rates
            b.edge_index = b.edge_index[:, keep]
            if b.edge_attr is not None:
                b.edge_attr = b.edge_attr[keep]

        if do_node:
            keep_p    = max(0.0, (1.0 - node_drop) * subgraph)
            node_keep = torch.rand(num_nodes, device=device) < keep_p

            if b.num_graphs > 1 or not bool(node_keep.any()):
                kept_per_graph = torch.bincount(
                    b.batch[node_keep], minlength=b.num_graphs)
                empty = (kept_per_graph == 0).nonzero(as_tuple=False).squeeze(1)
                if empty.numel() > 0:
                    first_idx = torch.full((b.num_graphs,), num_nodes,
                                           dtype=torch.long, device=device)
                    idx = torch.arange(num_nodes, device=device)
                    first_idx.scatter_reduce_(0, b.batch, idx, reduce='amin',
                                              include_self=True)
                    pick = (idx == first_idx[b.batch]) & \
                           (kept_per_graph[b.batch] == 0)
                    node_keep |= pick

            s, d  = b.edge_index
            ekeep = node_keep[s] & node_keep[d]

            b.edge_index = b.edge_index[:, ekeep]
            if b.edge_attr is not None and b.edge_attr.numel() > 0:
                b.edge_attr = b.edge_attr[ekeep]
            b.x     = b.x[node_keep]
            b.batch = b.batch[node_keep]

            remap = torch.full((num_nodes,), -1, dtype=torch.long, device=device)
            n_keep = int(node_keep.sum())
            remap[node_keep] = torch.arange(n_keep, device=device)
            if b.edge_index.numel() > 0:
                b.edge_index = remap[b.edge_index]

        if eff_feat > 0.0:
            b.x.masked_fill_(torch.rand(b.x.shape, device=device) < eff_feat, 0.0)
            if b.edge_attr is not None and b.edge_attr.numel() > 0:
                b.edge_attr.masked_fill_(
                    torch.rand(b.edge_attr.shape, device=device) < eff_feat, 0.0)

        if b.x.size(0) == 0:
            b.x = torch.zeros(1, batch.x.size(1),
                              device=device, dtype=batch.x.dtype)
            b.batch = torch.zeros(1, dtype=torch.long, device=device)
        if b.edge_index.size(1) == 0:
            b.edge_index = torch.empty(2, 0, device=device, dtype=torch.long)
            if b.edge_attr is not None:
                b.edge_attr = torch.empty(0, b.edge_attr.size(-1),
                                          device=device, dtype=b.edge_attr.dtype)
        return b

    b2 = _one(batch.clone())
    b1 = _one(batch)
    return b1, b2


def _collate_graphs_only(batch):
    return Batch.from_data_list(batch)


def autocast_ctx():
    if DEVICE.type != 'cuda':
        return contextlib.nullcontext()
    try:
        return torch.amp.autocast(device_type='cuda', dtype=torch.float16)
    except Exception:
        return torch.cuda.amp.autocast(enabled=True)

print("graph utilities ready")

graph utilities ready


## Cell 4 — Models (GATEncoder / BYOL / MoCo)

In [4]:
class GATEncoder(nn.Module):
    """Architecture MUST match the SSL notebooks exactly, otherwise
    the frozen encoder checkpoints will not load."""
    def __init__(self, in_ch, hid, heads, layers, edge_dim):
        super().__init__()
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        self.convs.append(GATConv(in_ch, hid, heads=heads, edge_dim=edge_dim,
                                  concat=True, dropout=0.0))
        self.bns.append(nn.BatchNorm1d(hid * heads))
        for _ in range(layers - 2):
            self.convs.append(GATConv(hid * heads, hid, heads=heads,
                                      edge_dim=edge_dim, concat=True, dropout=0.0))
            self.bns.append(nn.BatchNorm1d(hid * heads))
        self.convs.append(GATConv(hid * heads, hid, heads=1, edge_dim=edge_dim,
                                  concat=False, dropout=0.0))
        self.bns.append(nn.BatchNorm1d(hid))
        self.edge_dim = edge_dim
        self.pool_gate = nn.Sequential(
            nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))

    def forward(self, x, ei, ea=None, b=None, return_nodes=False):
        for i, c in enumerate(self.convs):
            x = c(x, ei, edge_attr=ea) if (ea is not None and self.edge_dim > 0) \
                else c(x, ei)
            x = F.relu(self.bns[i](x))
        h = GlobalAttention(self.pool_gate)(x, b)
        if return_nodes:
            return x, h
        return h


class BYOLModel(nn.Module):
    def __init__(self, in_ch, hid, heads, layers, edge_dim, proj_dim,
                 predictor_hid=256):
        super().__init__()
        self.encoder   = GATEncoder(in_ch, hid, heads, layers, edge_dim)
        self.projector = nn.Sequential(
            nn.Linear(hid, hid), nn.BatchNorm1d(hid), nn.ReLU(),
            nn.Linear(hid, proj_dim),
        )
        self.predictor = nn.Sequential(
            nn.Linear(proj_dim, predictor_hid), nn.BatchNorm1d(predictor_hid), nn.ReLU(),
            nn.Linear(predictor_hid, proj_dim),
        )

    def forward(self, x, ei, ea=None, b=None, predict=True, return_nodes=False):
        if return_nodes:
            h_nodes, h = self.encoder(x, ei, ea, b, return_nodes=True)
        else:
            h = self.encoder(x, ei, ea, b); h_nodes = None
        z = self.projector(h)
        if predict:
            p = self.predictor(z)
            return (h_nodes, h, z, p) if return_nodes else (h, z, p)
        return (h_nodes, h, z) if return_nodes else (h, z)


class MoCoModel(nn.Module):
    def __init__(self, in_ch, hid, heads, layers, edge_dim, proj_dim,
                 queue_size=QUEUE_SIZE, temperature=TEMPERATURE):
        super().__init__()
        self.encoder   = GATEncoder(in_ch, hid, heads, layers, edge_dim)
        self.projector = nn.Sequential(
            nn.Linear(hid, hid), nn.BatchNorm1d(hid), nn.ReLU(),
            nn.Linear(hid, proj_dim),
        )
        self.queue_size  = queue_size
        self.temperature = temperature
        self.register_buffer('queue',        torch.zeros(queue_size, proj_dim))
        self.register_buffer('queue_ptr',    torch.zeros(1, dtype=torch.long))
        self.register_buffer('queue_filled', torch.zeros(1, dtype=torch.long))

    def forward(self, x, ei, ea=None, b=None, return_nodes=False):
        if return_nodes:
            h_nodes, h = self.encoder(x, ei, ea, b, return_nodes=True)
            z = self.projector(h)
            return h_nodes, h, z
        h = self.encoder(x, ei, ea, b)
        z = self.projector(h)
        return h, z

    @torch.no_grad()
    def dequeue_and_enqueue(self, keys):
        keys = keys.detach(); bs = keys.shape[0]; ptr = int(self.queue_ptr)
        if bs >= self.queue_size:
            self.queue.copy_(keys[-self.queue_size:])
            self.queue_ptr[0] = 0; self.queue_filled[0] = self.queue_size
            return
        if ptr + bs <= self.queue_size:
            self.queue[ptr:ptr + bs] = keys
        else:
            first = self.queue_size - ptr
            self.queue[ptr:] = keys[:first]
            self.queue[:bs - first] = keys[first:]
        self.queue_ptr[0] = (ptr + bs) % self.queue_size
        self.queue_filled[0] = min(int(self.queue_filled.item()) + bs,
                                   self.queue_size)


class MaskedReconDecoder(nn.Module):
    def __init__(self, hid, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(),
                                 nn.Linear(hid, out_dim))
    def forward(self, h_nodes):
        return self.net(h_nodes)

print("models ready")

models ready


## Cell 5 — Losses & EMA helpers

In [5]:
@torch.no_grad()
def update_momentum_encoder(target, online, m, skip_queue=True):
    t_params = list(target.parameters()); o_params = list(online.parameters())
    torch._foreach_mul_(t_params, m)
    torch._foreach_add_(t_params, o_params, alpha=1 - m)
    online_bufs = dict(online.named_buffers())
    for name, tb in target.named_buffers():
        ob = online_bufs.get(name, None)
        if ob is None or tb.shape != ob.shape: continue
        if skip_queue and name in ('queue', 'queue_ptr', 'queue_filled'): continue
        if not tb.is_floating_point():
            if name.endswith('num_batches_tracked'):
                tb.data.copy_(ob.data)
            continue
        tb.data.mul_(m).add_(ob.data, alpha=1 - m)


def byol_momentum_schedule(step, total_steps, base=BYOL_MOMENTUM_BASE,
                           final=BYOL_MOMENTUM_FINAL):
    progress = min(step / max(total_steps - 1, 1), 1.0)
    return final - (final - base) * (math.cos(math.pi * progress) + 1.0) / 2.0


def byol_loss(p, z):
    p = F.normalize(p, dim=-1, eps=1e-8)
    z = F.normalize(z.detach(), dim=-1, eps=1e-8)
    return 2.0 - 2.0 * (p * z).sum(dim=-1).mean()


def info_nce_loss(q, k_pos, k_neg, temperature, use_queue=True):
    l_pos = (q * k_pos).sum(-1, keepdim=True)
    if use_queue and k_neg is not None and k_neg.size(0) > 0:
        l_neg  = q @ k_neg.t()
        logits = torch.cat([l_pos, l_neg], dim=1) / temperature
    else:
        logits = (q @ k_pos.t()) / temperature
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, labels)

print("losses ready")

losses ready


## Cell 6 — Probe / embed / metric helpers

In [6]:
@torch.inference_mode()
def embed_graphs(encoder, graphs, device, batch_size=64):
    encoder.eval(); chunks = []; i = 0; cur_bs = batch_size
    if len(graphs) == 0:
        return np.zeros((0, HIDDEN_DIM), dtype=np.float32)
    while i < len(graphs):
        chunk = graphs[i:i + cur_bs]
        try:
            batch = Batch.from_data_list(chunk).to(device)
            with autocast_ctx():
                out = encoder(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            if isinstance(out, tuple): out = out[-1]
            arr = out.float().cpu().numpy()
            arr = np.nan_to_num(arr, nan=0.0, posinf=100.0, neginf=-100.0)
            chunks.append(arr)
            del batch, out
            i += cur_bs
            if device.type == 'cuda': torch.cuda.empty_cache()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and cur_bs > 1:
                torch.cuda.empty_cache(); cur_bs = max(1, cur_bs // 2); continue
            raise
    return np.vstack(chunks)


def _optimal_threshold(prob, y):
    thrs = np.linspace(0.05, 0.95, 91)
    best_t, best_f1 = 0.5, -1.0
    for t in thrs:
        f1 = f1_score(y, (prob >= t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    return float(best_t), float(best_f1)


def _safe_f1(y_true, y_pred, average):
    try: return float(f1_score(y_true, y_pred, average=average, zero_division=0))
    except Exception: return 0.0


def frozen_linear_probe(X_train, y_train, X_val, y_val, X_test, y_test):
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=100.0, neginf=-100.0)
    X_val   = np.nan_to_num(X_val,   nan=0.0, posinf=100.0, neginf=-100.0)
    X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=100.0, neginf=-100.0)

    empty = {'val_roc': 0.0, 'test_roc': 0.0, 'test_pr': 0.0,
             'test_f1_binary': 0.0, 'test_macro_f1': 0.0,
             'test_weighted_f1': 0.0, 'val_macro_f1': 0.0,
             'test_fpr': 0.0, 'test_fnr': 0.0, 'threshold': 0.5,
             'test_knn_roc': 0.0, 'val_silhouette': 0.0}
    out = {}
    if len(set(y_train)) < 2 or len(y_train) < 4:
        out.update(empty); return out

    scaler = StandardScaler().fit(X_train)
    Xtr, Xva, Xte = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)

    lr = LogisticRegression(max_iter=3000, class_weight='balanced',
                            n_jobs=-1).fit(Xtr, y_train)
    prob_val  = lr.predict_proba(Xva)[:, 1]
    prob_test = lr.predict_proba(Xte)[:, 1]
    thr, _ = _optimal_threshold(prob_val, y_val)
    pred_test = (prob_test >= thr).astype(int)

    out['val_roc']  = float(roc_auc_score(y_val, prob_val))
    out['test_roc'] = float(roc_auc_score(y_test, prob_test))
    out['test_pr']  = float(average_precision_score(y_test, prob_test))
    out['test_f1_binary']   = _safe_f1(y_test, pred_test, 'binary')
    out['test_macro_f1']    = _safe_f1(y_test, pred_test, 'macro')
    out['test_weighted_f1'] = _safe_f1(y_test, pred_test, 'weighted')
    out['val_macro_f1']     = _safe_f1(y_val,  (prob_val >= thr).astype(int), 'macro')

    cm  = confusion_matrix(y_test, pred_test, labels=[0, 1])
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    out['test_fpr'] = float(cmn[0, 1]); out['test_fnr'] = float(cmn[1, 0])
    out['threshold'] = float(thr)

    try:
        knn = KNeighborsClassifier(n_neighbors=KNN_K, weights='distance',
                                   n_jobs=-1).fit(Xtr, y_train)
        out['test_knn_roc'] = float(roc_auc_score(y_test, knn.predict_proba(Xte)[:, 1]))
    except Exception:
        out['test_knn_roc'] = 0.0

    if len(set(y_val)) > 1 and len(X_val) > 3:
        try: out['val_silhouette'] = float(silhouette_score(X_val, y_val, metric='cosine'))
        except Exception: out['val_silhouette'] = 0.0
    else:
        out['val_silhouette'] = 0.0
    return out

print("probe utilities ready")

probe utilities ready


## Cell 7 — Dataset loader with pluggable norm stats

The three cross-dataset protocols differ only in *which* mean/std gets applied. This loader takes a stats dict and returns normalised graphs plus binary labels.

In [7]:
BUNDLES = {}

def load_dataset_bundle(dataset_name, norm_stats=None, cache_key=None):
    """
    norm_stats : None         -> use dataset's own norm_stats.json
                 dict         -> use provided stats (must have node_mean/node_std/
                                 edge_mean/edge_std, all lists of floats)
    cache_key  : unique key for the in-memory cache; required when norm_stats != None
    """
    DATA_DIR = DATASET_CONFIG[dataset_name]
    if cache_key is None:
        cache_key = f"{dataset_name}__self" if norm_stats is None else None
    if cache_key and cache_key in BUNDLES:
        return BUNDLES[cache_key]

    def load_graphs(split):
        with open(os.path.join(DATA_DIR, f"{split}_graphs.pkl"), 'rb') as f:
            return pickle.load(f)

    train_graphs = [cap_graph(g) for g in load_graphs('train')]
    val_graphs   = [cap_graph(g) for g in load_graphs('val')]
    test_graphs  = [cap_graph(g) for g in load_graphs('test')]

    # ---- binary labels ----
    train_bin = np.array([0 if dominant_family(g) == 'Benign' else 1
                          for g in train_graphs], dtype=np.int64)
    val_bin   = np.array([0 if dominant_family(g) == 'Benign' else 1
                          for g in val_graphs],   dtype=np.int64)
    test_bin  = np.array([0 if dominant_family(g) == 'Benign' else 1
                          for g in test_graphs],  dtype=np.int64)

    # ---- normalisation stats ----
    if norm_stats is None:
        with open(os.path.join(DATA_DIR, "norm_stats.json")) as f:
            ns = json.load(f)
    else:
        with open(os.path.join(DATA_DIR, "norm_stats.json")) as f:
            base = json.load(f)
        ns = {**base,
              'node_mean': norm_stats['node_mean'], 'node_std': norm_stats['node_std'],
              'edge_mean': norm_stats['edge_mean'], 'edge_std': norm_stats['edge_std']}

    N_CONT    = ns["n_continuous_edge_feats"]
    node_mean = torch.tensor(ns["node_mean"], dtype=torch.float)
    node_std  = torch.tensor(ns["node_std"],  dtype=torch.float)
    edge_mean = torch.tensor(ns["edge_mean"], dtype=torch.float)
    edge_std  = torch.tensor(ns["edge_std"],  dtype=torch.float)

    def normalise(graphs):
        for g in graphs:
            g.x = torch.clamp((g.x - node_mean) / (node_std + 1e-8), -10, 10)
            if g.edge_attr is not None and g.edge_attr.size(0) > 0:
                if g.edge_attr.dim() > 2:
                    g.edge_attr = g.edge_attr.view(-1, g.edge_attr.size(-1))
                c, p = g.edge_attr[:, :N_CONT], g.edge_attr[:, N_CONT:]
                c = torch.clamp((c - edge_mean) / (edge_std + 1e-8), -10, 10)
                g.edge_attr = torch.cat([c, p], dim=1)
            if g.edge_attr is None or g.edge_attr.size(0) == 0:
                g.edge_attr = torch.zeros((g.edge_index.size(1), N_CONT), dtype=torch.float)
            g.x = torch.nan_to_num(g.x, nan=0.0, posinf=0.0, neginf=0.0)
            g.edge_attr = torch.nan_to_num(g.edge_attr, nan=0.0,
                                           posinf=0.0, neginf=0.0)

    normalise(train_graphs); normalise(val_graphs); normalise(test_graphs)

    # ---- SSL graphs (labels stripped) ----
    ssl_graphs = []
    for g in train_graphs:
        gc_ = g.clone()
        for a in ['y', 'edge_y', 'y_multihot', 'attack_label']:
            if hasattr(gc_, a): delattr(gc_, a)
        ssl_graphs.append(gc_)

    # ---- probe-train subsample ----
    if len(train_graphs) > PROBE_TRAIN_MAX:
        rng = np.random.default_rng(SEED)
        idx = rng.choice(len(train_graphs), PROBE_TRAIN_MAX, replace=False)
        probe_train_graphs = [train_graphs[i] for i in idx]
        probe_train_bin    = train_bin[idx]
    else:
        probe_train_graphs = train_graphs
        probe_train_bin    = train_bin

    samp     = ssl_graphs[0] if ssl_graphs else train_graphs[0]
    node_dim = samp.x.size(1)
    edge_dim = samp.edge_attr.size(1)

    bundle = dict(
        name=dataset_name, DATA_DIR=DATA_DIR,
        train_graphs=train_graphs, val_graphs=val_graphs, test_graphs=test_graphs,
        train_bin=train_bin, val_bin=val_bin, test_bin=test_bin,
        ssl_graphs=ssl_graphs,
        probe_train_graphs=probe_train_graphs, probe_train_bin=probe_train_bin,
        N_CONT=N_CONT, node_dim=node_dim, edge_dim=edge_dim,
    )
    if cache_key:
        BUNDLES[cache_key] = bundle
    return bundle


def load_self_stats(dataset_name):
    with open(os.path.join(DATASET_CONFIG[dataset_name], 'norm_stats.json')) as f:
        return json.load(f)

print("loader ready")

loader ready


## Cell 8 — Global norm stats

Concatenate raw (un-normalised) train features across every dataset and compute a single mean/std. This is what makes source- and joint-dataset transfer comparable.

In [8]:
GLOBAL_NS_PATH = 'global_norm_stats.json'

def compute_global_norm_stats(save_path=GLOBAL_NS_PATH):
    Xs, Es, per_ds = [], [], {}
    for ds, d in DATASET_CONFIG.items():
        with open(os.path.join(d, 'train_graphs.pkl'), 'rb') as f:
            train = pickle.load(f)
        for g in train:
            Xs.append(g.x.numpy())
            if g.edge_attr is not None and g.edge_attr.shape[0] > 0:
                ea = g.edge_attr.numpy()
                if ea.ndim == 3: ea = ea.reshape(-1, ea.shape[-1])
                Es.append(ea)
        per_ds[ds] = len(train)
        print(f"  {ds}: {len(train)} train graphs")

    X = np.concatenate(Xs, 0).astype(np.float32)
    E = np.concatenate(Es, 0).astype(np.float32)
    print(f"  total: {X.shape[0]:,} node rows  {E.shape[0]:,} edge rows")

    with open(os.path.join(list(DATASET_CONFIG.values())[0], 'norm_stats.json')) as f:
        base = json.load(f)

    stats = {
        'node_mean': X.mean(0).tolist(),
        'node_std':  (X.std(0)  + 1e-8).tolist(),
        'edge_mean': E.mean(0).tolist(),
        'edge_std':  (E.std(0)  + 1e-8).tolist(),
        'n_continuous_edge_feats': base['n_continuous_edge_feats'],
        'node_cols': base['node_cols'], 'edge_cols': base['edge_cols'],
        'source_datasets': list(DATASET_CONFIG.keys()),
        'n_node_rows': int(X.shape[0]), 'n_edge_rows': int(E.shape[0]),
        'per_dataset_train_graphs': per_ds,
    }
    with open(save_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"saved {save_path}")
    return stats

if os.path.isfile(GLOBAL_NS_PATH):
    with open(GLOBAL_NS_PATH) as f: GLOBAL_NS = json.load(f)
    print(f"[cache] loaded {GLOBAL_NS_PATH}")
else:
    print("Computing global norm stats ...")
    GLOBAL_NS = compute_global_norm_stats()
print(f"  node_cols : {len(GLOBAL_NS['node_cols'])}")
print(f"  edge_cols : {len(GLOBAL_NS['edge_cols'])}")

Computing global norm stats ...
  gotham: 2772 train graphs
  nf_bot: 5170 train graphs
  nf_ton: 23404 train graphs
  total: 649,559 node rows  25,348,103 edge rows
saved global_norm_stats.json
  node_cols : 7
  edge_cols : 10


## Cell 9 — Load source-pretrained encoder

In [9]:
def encoder_path_for(dataset_name, ssl_method):
    d = os.path.join(DATASET_CONFIG[dataset_name], f"ssl_checkpoints_{ssl_method}")
    return os.path.join(d, 'ssl_encoder_best.pth')


def load_encoder(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    enc = GATEncoder(ckpt['node_dim'], ckpt['hidden_dim'], ckpt['num_heads'],
                     ckpt['num_layers'], ckpt['edge_dim']).to(DEVICE)
    enc.load_state_dict(ckpt['encoder_state_dict'])
    enc.eval()
    return enc, ckpt


print("Per-dataset SSL encoder checkpoints:")
for ds in DATASET_CONFIG:
    for ssl in SSL_METHODS:
        p = encoder_path_for(ds, ssl)
        ok = "OK" if os.path.isfile(p) else "MISSING"
        print(f"  {ds:<8} {ssl:<5} {ok}   {p}")

Per-dataset SSL encoder checkpoints:
  gotham   byol  OK   Gotham_processed_features_10s/graphs\ssl_checkpoints_byol\ssl_encoder_best.pth
  gotham   moco  OK   Gotham_processed_features_10s/graphs\ssl_checkpoints_moco\ssl_encoder_best.pth
  nf_bot   byol  OK   NFBoTIoT_v3_10s/graphs\ssl_checkpoints_byol\ssl_encoder_best.pth
  nf_bot   moco  OK   NFBoTIoT_v3_10s/graphs\ssl_checkpoints_moco\ssl_encoder_best.pth
  nf_ton   byol  OK   NFToNIoT_v3_10s/graphs\ssl_checkpoints_byol\ssl_encoder_best.pth
  nf_ton   moco  OK   NFToNIoT_v3_10s/graphs\ssl_checkpoints_moco\ssl_encoder_best.pth


## Cell 10 — Protocol A · Zero-shot probe

Frozen source encoder, target labels only for the probe. Three normalisation modes.

In [10]:
def zero_shot_probe(source_ds, target_ds, ssl_method,
                    norm_protocol='source'):
    """
    norm_protocol:
      'self'   -> use target's own stats (in-domain baseline)
      'source' -> apply source's stats to target
      'global' -> apply global stats to target
    """
    src_ckpt = encoder_path_for(source_ds, ssl_method)
    if not os.path.isfile(src_ckpt):
        return None

    enc, ckpt = load_encoder(src_ckpt)

    if norm_protocol == 'self':
        tgt = load_dataset_bundle(target_ds,
                                  cache_key=f"{target_ds}__self")
    elif norm_protocol == 'source':
        stats = load_self_stats(source_ds)
        tgt = load_dataset_bundle(target_ds, norm_stats=stats,
                                  cache_key=f"{target_ds}__src_{source_ds}")
    elif norm_protocol == 'global':
        tgt = load_dataset_bundle(target_ds, norm_stats=GLOBAL_NS,
                                  cache_key=f"{target_ds}__global")
    else:
        raise ValueError(norm_protocol)

    # guard: target must have both classes in probe train
    if len(set(tgt['probe_train_bin'])) < 2:
        del enc; gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        return None

    X_tr = embed_graphs(enc, tgt['probe_train_graphs'], DEVICE)
    X_va = embed_graphs(enc, tgt['val_graphs'],         DEVICE)
    X_te = embed_graphs(enc, tgt['test_graphs'],        DEVICE)

    res = frozen_linear_probe(
        X_tr, tgt['probe_train_bin'],
        X_va, tgt['val_bin'],
        X_te, tgt['test_bin'],
    )
    res['source_dim'] = int(ckpt['node_dim'])
    res['target_dim'] = int(tgt['node_dim'])
    res['src_epoch']  = int(ckpt.get('epoch', -1))
    res['src_val_roc']= float(ckpt.get('val_roc_auc', float('nan')))

    del enc
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return res

print("zero_shot_probe ready")

zero_shot_probe ready


## Cell 11 — Run Protocol A over every (source, target, ssl, norm) cell

Runs are cached to `cross_dataset_probe_results.csv`. Delete that file to force a re-run.

In [13]:
PROBE_CSV = 'cross_dataset_probe_results.csv'

if os.path.isfile(PROBE_CSV):
    df_probe = pd.read_csv(PROBE_CSV)
    print(f"[cache] loaded {PROBE_CSV}  ({len(df_probe)} rows)")
else:
    rows = []
    for ssl in SSL_METHODS:
        for src in DATASET_CONFIG:
            if not os.path.isfile(encoder_path_for(src, ssl)):
                print(f"  [skip] no encoder for {src}/{ssl}")
                continue
            for tgt in DATASET_CONFIG:
                protocols = ['self'] if src == tgt else ['source', 'global']
                for proto in protocols:
                    tag = f"{ssl:>4} | {src:>6} -> {tgt:<6} | {proto:<6}"
                    try:
                        r = zero_shot_probe(src, tgt, ssl, norm_protocol=proto)
                    except Exception as e:
                        print(f"  {tag}  FAILED: {e}")
                        continue
                    if r is None:
                        print(f"  {tag}  (skipped)")
                        continue
                    rows.append({
                        'source': src, 'target': tgt, 'ssl': ssl, 'norm': proto,
                        'ROC': r['test_roc'], 'PR': r['test_pr'],
                        'F1':  r['test_f1_binary'],
                        'FPR': r['test_fpr'], 'FNR': r['test_fnr'],
                        'threshold': r['threshold'],
                        'val_ROC': r['val_roc'], 'val_mF1': r['val_macro_f1'],
                        'knn_ROC': r['test_knn_roc'],
                        'src_epoch': r['src_epoch'],
                        'src_val_roc': r['src_val_roc'],
                    })
                    print(f"  {tag}  ROC {r['test_roc']:.4f}  F1 {r['test_f1_binary']:.4f}  PR {r['test_pr']:.4f}   FPR {r['test_fpr']:.4f}  FNR {r['test_fnr']:.4f}  val_ROC {r['val_roc']:.4f}  val_mF1 {r['val_macro_f1']:.4f}  knn_ROC {r['test_knn_roc']:.4f}")
    df_probe = pd.DataFrame(rows)
    df_probe.to_csv(PROBE_CSV, index=False)
    print(f"\nSaved {PROBE_CSV}  ({len(df_probe)} rows)")

if not df_probe.empty:
    print("\n=== Zero-shot probe — full grid ===")
    cols = ['ssl','source','target','norm','ROC','PR','F1','FPR','FNR']
    print(df_probe[cols].sort_values(
        ['ssl','target','source','norm']).to_string(index=False))

  byol | gotham -> gotham | self    ROC 0.9739  F1 0.8836  PR 0.8817   FPR 0.0507  FNR 0.0786  val_ROC 0.9865  val_mF1 0.9595  knn_ROC 0.9801
  byol | gotham -> nf_bot | source  ROC 0.7868  F1 0.9444  PR 0.9347   FPR 0.3935  FNR 0.0482  val_ROC 0.7947  val_mF1 0.7701  knn_ROC 0.9013
  byol | gotham -> nf_bot | global  ROC 0.8918  F1 0.9425  PR 0.9790   FPR 0.3742  FNR 0.0545  val_ROC 0.7933  val_mF1 0.7669  knn_ROC 0.9562
  byol | gotham -> nf_ton | source  ROC 0.3647  F1 0.3431  PR 0.8239   FPR 0.5120  FNR 0.7811  val_ROC 0.4451  val_mF1 0.4152  knn_ROC 0.2592
  byol | gotham -> nf_ton | global  ROC 0.2856  F1 0.3381  PR 0.8084   FPR 0.5558  FNR 0.7840  val_ROC 0.2640  val_mF1 0.3467  knn_ROC 0.1947
  byol | nf_bot -> gotham | source  ROC 0.9601  F1 0.6802  PR 0.8763   FPR 0.0507  FNR 0.4000  val_ROC 0.9842  val_mF1 0.9701  knn_ROC 0.9275
  byol | nf_bot -> gotham | global  ROC 0.9594  F1 0.8919  PR 0.8441   FPR 0.0529  FNR 0.0571  val_ROC 0.9806  val_mF1 0.9509  knn_ROC 0.9052
  byol

## Cell 12 — Transfer-gap table (Protocol A)

In [14]:
def transfer_gap_table(df_probe, metric='ROC'):
    """Δ = in-domain (target's own stats)  −  transfer."""
    if df_probe.empty:
        return pd.DataFrame()

    in_dom = (df_probe[df_probe['source'] == df_probe['target']]
              .set_index(['ssl','target','norm'])[metric])

    out = df_probe[df_probe['source'] != df_probe['target']].copy()
    out['in_domain'] = out.apply(
        lambda r: in_dom.get((r['ssl'], r['target'], 'self'), np.nan), axis=1)
    out['transfer_gap'] = out['in_domain'] - out[metric]
    out = out[['ssl','source','target','norm',
               metric,'in_domain','transfer_gap']]
    out = out.sort_values(['ssl','target','source','norm']).reset_index(drop=True)
    return out

gap = transfer_gap_table(df_probe, 'ROC')
if not gap.empty:
    print("=== Transfer gap (ROC) — smaller is better ===")
    print(gap.to_string(index=False))
    gap.to_csv('cross_dataset_transfer_gap.csv', index=False)

    # aggregate view: mean gap per (ssl, norm)
    print("\n=== Mean transfer gap per SSL method × norm protocol ===")
    agg = (gap.groupby(['ssl','norm'])['transfer_gap']
              .agg(['mean','std','min','max','count'])
              .round(4))
    print(agg.to_string())

=== Transfer gap (ROC) — smaller is better ===
 ssl source target   norm      ROC  in_domain  transfer_gap
byol nf_bot gotham global 0.959361   0.973914      0.014553
byol nf_bot gotham source 0.960132   0.973914      0.013782
byol nf_ton gotham global 0.973946   0.973914     -0.000031
byol nf_ton gotham source 0.982096   0.973914     -0.008181
byol gotham nf_bot global 0.891790   0.880571     -0.011219
byol gotham nf_bot source 0.786755   0.880571      0.093816
byol nf_ton nf_bot global 0.749861   0.880571      0.130709
byol nf_ton nf_bot source 0.742034   0.880571      0.138537
byol gotham nf_ton global 0.285606   0.614717      0.329111
byol gotham nf_ton source 0.364723   0.614717      0.249994
byol nf_bot nf_ton global 0.374974   0.614717      0.239744
byol nf_bot nf_ton source 0.322282   0.614717      0.292435
moco nf_bot gotham global 0.973505   0.969965     -0.003540
moco nf_bot gotham source 0.965812   0.969965      0.004154
moco nf_ton gotham global 0.960038   0.969965      0.

## Cell 13 — Protocol C · Joint pretraining bundle

Concatenate every dataset's `ssl_graphs` into one pool. Because we want a checkpoint-selection signal during joint training, we also concatenate the val/test splits.

In [15]:
def build_joint_bundle(norm_protocol='global', tag='joint'):
    """
    norm_protocol : 'self'   -> each dataset keeps its own stats
                    'global' -> every dataset uses GLOBAL_NS
    """
    ssl_pool, tr_pool, tr_bin_pool = [], [], []
    va_pool,  va_bin_pool = [], []
    te_pool,  te_bin_pool = [], []

    for ds in DATASET_CONFIG:
        if norm_protocol == 'global':
            b = load_dataset_bundle(ds, norm_stats=GLOBAL_NS,
                                    cache_key=f"{ds}__global")
        else:
            b = load_dataset_bundle(ds, cache_key=f"{ds}__self")
        ssl_pool.extend(b['ssl_graphs'])
        tr_pool.extend(b['train_graphs']);  tr_bin_pool.extend(b['train_bin'].tolist())
        va_pool.extend(b['val_graphs']);    va_bin_pool.extend(b['val_bin'].tolist())
        te_pool.extend(b['test_graphs']);   te_bin_pool.extend(b['test_bin'].tolist())

    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(ssl_pool))
    ssl_pool = [ssl_pool[i] for i in order]

    tr_bin  = np.array(tr_bin_pool,  dtype=np.int64)
    va_bin  = np.array(va_bin_pool,  dtype=np.int64)
    te_bin  = np.array(te_bin_pool,  dtype=np.int64)

    # probe-train subsample from the joint train pool
    if len(tr_pool) > PROBE_TRAIN_MAX:
        idx = rng.choice(len(tr_pool), PROBE_TRAIN_MAX, replace=False)
        probe_tr  = [tr_pool[i] for i in idx]
        probe_bin = tr_bin[idx]
    else:
        probe_tr  = tr_pool; probe_bin = tr_bin

    samp = ssl_pool[0]
    bundle = dict(
        name=tag,
        ssl_graphs=ssl_pool,
        train_graphs=tr_pool, train_bin=tr_bin,
        val_graphs=va_pool,   val_bin=va_bin,
        test_graphs=te_pool,  test_bin=te_bin,
        probe_train_graphs=probe_tr, probe_train_bin=probe_bin,
        node_dim=samp.x.size(1), edge_dim=samp.edge_attr.size(1),
        N_CONT=GLOBAL_NS['n_continuous_edge_feats'],
        checkpoint_dir=f'ssl_checkpoints_{tag}',
        report_path=f'ssl_checkpoints_{tag}/report.json',
        encoder_path=f'ssl_checkpoints_{tag}/encoder_best.pth',
    )
    os.makedirs(bundle['checkpoint_dir'], exist_ok=True)
    print(f"joint bundle '{tag}': n_ssl={len(ssl_pool):,}  "
          f"n_train={len(tr_pool):,}  n_val={len(va_pool):,}  n_test={len(te_pool):,}")
    return bundle

print("joint-bundle builder ready")

joint-bundle builder ready


## Cell 14 — Joint BYOL pretraining

In [16]:
def ssl_loader(ssl_graphs, batch_size=BATCH_SIZE, num_workers=0):
    kw = dict(batch_size=batch_size, shuffle=True,
              collate_fn=_collate_graphs_only, num_workers=num_workers,
              pin_memory=(DEVICE.type == 'cuda'), drop_last=True)
    return DataLoader(ssl_graphs, **kw)


def _ssl_eval(enc, bundle, ssl_method):
    X_tr = embed_graphs(enc, bundle['probe_train_graphs'], DEVICE)
    X_va = embed_graphs(enc, bundle['val_graphs'],         DEVICE)
    X_te = embed_graphs(enc, bundle['test_graphs'],        DEVICE)
    return frozen_linear_probe(
        X_tr, bundle['probe_train_bin'],
        X_va, bundle['val_bin'],
        X_te, bundle['test_bin'])


def train_joint_byol(bundle, force_retrain=False):
    encoder_path = bundle['encoder_path']
    report_path  = bundle['report_path']

    if (not force_retrain) and os.path.isfile(encoder_path) and os.path.isfile(report_path):
        print(f"[skip] joint BYOL already trained ({encoder_path})")
        with open(report_path) as f: return json.load(f)

    print(f"\n{'='*70}\nJOINT BYOL · {bundle['name']}\n{'='*70}")
    print(f"  n_ssl={len(bundle['ssl_graphs']):,}  epochs={SSL_EPOCHS}")
    loader = ssl_loader(bundle['ssl_graphs'])

    node_dim, edge_dim = bundle['node_dim'], bundle['edge_dim']
    online = BYOLModel(node_dim, HIDDEN_DIM, NUM_HEADS, NUM_LAYERS, edge_dim,
                       PROJECTION_DIM, PREDICTOR_HID).to(DEVICE)
    target = copy.deepcopy(online).to(DEVICE)
    for p in target.parameters(): p.requires_grad = False

    recon_head = MaskedReconDecoder(HIDDEN_DIM, node_dim).to(DEVICE) \
                 if USE_MASKED_RECON else None

    params = list(online.parameters()) + (list(recon_head.parameters())
                                          if recon_head else [])
    opt = torch.optim.AdamW(params, lr=SSL_LR, weight_decay=SSL_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SSL_EPOCHS,
                                                       eta_min=1e-6)
    scaler = GradScaler()
    total_steps = SSL_EPOCHS * max(len(loader), 1)
    gstep = 0
    best_roc, patience = -1.0, 0
    history = []

    for epoch in range(1, SSL_EPOCHS + 1):
        online.train(); target.eval()
        if recon_head: recon_head.train()
        L, LB, LR, n = 0.0, 0.0, 0.0, 0
        t0 = time.time()
        opt.zero_grad(set_to_none=True)

        for bi, batch in enumerate(tqdm(loader, desc=f"E{epoch:03d}", leave=False)):
            batch = batch.to(DEVICE, non_blocking=True)
            b1, b2 = augment_batch(batch, JOINT_EDGE_DROP, JOINT_FEAT_MASK,
                                   JOINT_NODE_DROP, JOINT_SUBGRAPH)
            m = byol_momentum_schedule(gstep, total_steps)

            with autocast_ctx():
                h1, z1, p1 = online(b1.x, b1.edge_index, b1.edge_attr, b1.batch, predict=True)
                h2, z2, p2 = online(b2.x, b2.edge_index, b2.edge_attr, b2.batch, predict=True)
                with torch.no_grad():
                    _, z_t1 = target(b1.x, b1.edge_index, b1.edge_attr, b1.batch, predict=False)
                    _, z_t2 = target(b2.x, b2.edge_index, b2.edge_attr, b2.batch, predict=False)
                l_byol = 0.5 * (byol_loss(p1, z_t2.float()) + byol_loss(p2, z_t1.float()))

                l_rec = torch.zeros((), device=DEVICE)
                if USE_MASKED_RECON and recon_head is not None:
                    nm = torch.rand(b1.x.size(0), device=DEVICE) < RECON_MASK_RATIO
                    if nm.any():
                        xt = b1.x.clone(); xm = b1.x.clone(); xm[nm] = 0.0
                        hn, _ = online.encoder(xm, b1.edge_index, b1.edge_attr,
                                               b1.batch, return_nodes=True)
                        rec = recon_head(hn)
                        l_rec = F.mse_loss(rec[nm], xt[nm])
                loss = l_byol + RECON_WEIGHT * l_rec

            scaler.scale(loss).backward()
            if (bi + 1) % GRAD_ACCUM_STEPS == 0 or (bi + 1) == len(loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
                update_momentum_encoder(target, online, m, skip_queue=False)

            L += float(loss); LB += float(l_byol); LR += float(l_rec)
            n += 1; gstep += 1

            del b1, b2, loss, z1, z2, p1, p2, h1, h2, z_t1, z_t2, l_byol, l_rec
            if DEVICE.type == 'cuda' and (bi + 1) % 25 == 0:
                torch.cuda.empty_cache()

        sched.step()
        n = max(n, 1)
        dt = time.time() - t0

        if epoch % SSL_EVAL_EVERY == 0 or epoch == 1:
            res = _ssl_eval(online.encoder, bundle, 'byol')
            improved = res['val_roc'] > best_roc + 1e-4
            if improved:
                best_roc = res['val_roc']; patience = 0
                torch.save({
                    'encoder_state_dict': online.encoder.state_dict(),
                    'node_dim': node_dim, 'edge_dim': edge_dim,
                    'hidden_dim': HIDDEN_DIM, 'num_heads': NUM_HEADS, 'num_layers': NUM_LAYERS,
                    'n_cont': bundle['N_CONT'],
                    'epoch': epoch, 'val_roc_auc': float(res['val_roc']),
                    'ssl_objective': 'joint_byol_masked_recon',
                    'dataset': bundle['name'], 'window_sec': 10,
                }, encoder_path)
                marker = "*best*"
            else:
                patience += 1; marker = ""
            print(f"E{epoch:03d} | {dt:5.1f}s | loss {L/n:.4f} | byol {LB/n:.4f} | "
                  f"rec {LR/n:.4f} | val ROC {res['val_roc']:.4f} | "
                  f"test ROC {res['test_roc']:.4f} | F1 {res['test_f1_binary']:.4f} {marker}")
            history.append(dict(epoch=epoch, loss=L/n, byol=LB/n, recon=LR/n,
                                val_roc=res['val_roc'], test_roc=res['test_roc'],
                                test_f1=res['test_f1_binary']))
            if patience * SSL_EVAL_EVERY >= SSL_EARLY_STOP:
                print(f"early stop at ep {epoch}"); break
        else:
            print(f"E{epoch:03d} | {dt:5.1f}s | loss {L/n:.4f} | byol {LB/n:.4f} | rec {LR/n:.4f}")

        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    report = {'dataset': bundle['name'], 'ssl_objective': 'joint_byol_masked_recon',
              'n_ssl': len(bundle['ssl_graphs']), 'history': history,
              'config': {'hidden_dim': HIDDEN_DIM, 'num_heads': NUM_HEADS,
                         'num_layers': NUM_LAYERS, 'projection_dim': PROJECTION_DIM,
                         'batch_size': BATCH_SIZE, 'lr': SSL_LR, 'epochs': SSL_EPOCHS,
                         'edge_dropout': JOINT_EDGE_DROP, 'feat_mask': JOINT_FEAT_MASK,
                         'node_drop': JOINT_NODE_DROP, 'subgraph': JOINT_SUBGRAPH,
                         'use_masked_recon': USE_MASKED_RECON,
                         'recon_weight': RECON_WEIGHT,
                         'labels_used_in_ssl': False}}
    with open(report_path, 'w') as f: json.dump(report, f, indent=2)

    del online, target, recon_head, opt, sched, scaler
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return report

print("joint BYOL trainer ready")

joint BYOL trainer ready


## Cell 15 — Joint MoCo pretraining

In [17]:
def train_joint_moco(bundle, force_retrain=False):
    encoder_path = bundle['encoder_path'].replace('joint', 'joint_moco') \
                   if 'joint' in bundle['name'] else bundle['encoder_path']
    report_path  = bundle['report_path'].replace('joint', 'joint_moco') \
                   if 'joint' in bundle['name'] else bundle['report_path']
    os.makedirs(os.path.dirname(encoder_path), exist_ok=True)

    if (not force_retrain) and os.path.isfile(encoder_path) and os.path.isfile(report_path):
        print(f"[skip] joint MoCo already trained ({encoder_path})")
        with open(report_path) as f: return json.load(f)

    print(f"\n{'='*70}\nJOINT MoCo · {bundle['name']}\n{'='*70}")
    print(f"  n_ssl={len(bundle['ssl_graphs']):,}  epochs={SSL_EPOCHS}")
    loader = ssl_loader(bundle['ssl_graphs'])

    node_dim, edge_dim = bundle['node_dim'], bundle['edge_dim']
    online = MoCoModel(node_dim, HIDDEN_DIM, NUM_HEADS, NUM_LAYERS, edge_dim,
                       PROJECTION_DIM, QUEUE_SIZE, TEMPERATURE).to(DEVICE)
    target = copy.deepcopy(online).to(DEVICE)
    for p in target.parameters(): p.requires_grad = False

    recon_head = MaskedReconDecoder(HIDDEN_DIM, node_dim).to(DEVICE) \
                 if USE_MASKED_RECON else None

    params = list(online.parameters()) + (list(recon_head.parameters())
                                          if recon_head else [])
    opt = torch.optim.AdamW(params, lr=SSL_LR, weight_decay=SSL_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SSL_EPOCHS,
                                                       eta_min=1e-6)
    scaler = GradScaler()
    best_roc, patience = -1.0, 0
    history = []

    for epoch in range(1, SSL_EPOCHS + 1):
        online.train(); target.eval()
        if recon_head: recon_head.train()
        L, LN, LR, n = 0.0, 0.0, 0.0, 0
        t0 = time.time()
        opt.zero_grad(set_to_none=True)

        for bi, batch in enumerate(tqdm(loader, desc=f"E{epoch:03d}", leave=False)):
            batch = batch.to(DEVICE, non_blocking=True)
            b1, b2 = augment_batch(batch, JOINT_EDGE_DROP, JOINT_FEAT_MASK,
                                   JOINT_NODE_DROP, JOINT_SUBGRAPH)

            with autocast_ctx():
                nm = torch.rand(b1.x.size(0), device=DEVICE) < RECON_MASK_RATIO
                xt = b1.x.clone(); xm = b1.x.clone()
                if nm.any(): xm[nm] = 0.0

                h1n, h1, z1 = online(xm, b1.edge_index, b1.edge_attr, b1.batch,
                                     return_nodes=True)
                with torch.no_grad():
                    _, z2 = target(b2.x, b2.edge_index, b2.edge_attr, b2.batch)
                z1n = F.normalize(z1.float(), dim=-1)
                z2n = F.normalize(z2.float(), dim=-1)
                use_q = int(online.queue_filled.item()) >= max(1, BATCH_SIZE)
                l_nce = info_nce_loss(z1n, z2n, online.queue,
                                      online.temperature, use_queue=use_q)
                l_rec = torch.zeros((), device=DEVICE)
                if USE_MASKED_RECON and recon_head is not None and nm.any():
                    rec = recon_head(h1n.float())
                    l_rec = F.mse_loss(rec[nm], xt[nm])
                loss = l_nce + RECON_WEIGHT * l_rec

            scaler.scale(loss).backward()
            if (bi + 1) % GRAD_ACCUM_STEPS == 0 or (bi + 1) == len(loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)
                update_momentum_encoder(target, online, MOCO_MOMENTUM, skip_queue=True)
                with torch.no_grad():
                    _, z_enq = target(b2.x, b2.edge_index, b2.edge_attr, b2.batch)
                    z_enq = F.normalize(z_enq.float(), dim=-1)
                online.dequeue_and_enqueue(z_enq)

            L += float(loss); LN += float(l_nce); LR += float(l_rec); n += 1
            del b1, b2, loss, z1, z2, h1, h1n, z1n, z2n, l_nce, l_rec, nm, xt, xm
            if DEVICE.type == 'cuda' and (bi + 1) % 25 == 0:
                torch.cuda.empty_cache()

        sched.step()
        n = max(n, 1); dt = time.time() - t0

        if epoch % SSL_EVAL_EVERY == 0 or epoch == 1:
            res = _ssl_eval(online.encoder, bundle, 'moco')
            improved = res['val_roc'] > best_roc + 1e-4
            if improved:
                best_roc = res['val_roc']; patience = 0
                torch.save({
                    'encoder_state_dict': online.encoder.state_dict(),
                    'node_dim': node_dim, 'edge_dim': edge_dim,
                    'hidden_dim': HIDDEN_DIM, 'num_heads': NUM_HEADS, 'num_layers': NUM_LAYERS,
                    'n_cont': bundle['N_CONT'],
                    'epoch': epoch, 'val_roc_auc': float(res['val_roc']),
                    'ssl_objective': 'joint_moco_masked_recon',
                    'dataset': bundle['name'], 'window_sec': 10,
                }, encoder_path)
                marker = "*best*"
            else:
                patience += 1; marker = ""
            print(f"E{epoch:03d} | {dt:5.1f}s | loss {L/n:.4f} | nce {LN/n:.4f} | "
                  f"rec {LR/n:.4f} | val ROC {res['val_roc']:.4f} | "
                  f"test ROC {res['test_roc']:.4f} | F1 {res['test_f1_binary']:.4f} {marker}")
            history.append(dict(epoch=epoch, loss=L/n, nce=LN/n, recon=LR/n,
                                val_roc=res['val_roc'], test_roc=res['test_roc'],
                                test_f1=res['test_f1_binary']))
            if patience * SSL_EVAL_EVERY >= SSL_EARLY_STOP:
                print(f"early stop at ep {epoch}"); break
        else:
            print(f"E{epoch:03d} | {dt:5.1f}s | loss {L/n:.4f} | nce {LN/n:.4f} | rec {LR/n:.4f}")

        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    report = {'dataset': bundle['name'], 'ssl_objective': 'joint_moco_masked_recon',
              'n_ssl': len(bundle['ssl_graphs']), 'history': history,
              'config': {'hidden_dim': HIDDEN_DIM, 'num_heads': NUM_HEADS,
                         'num_layers': NUM_LAYERS, 'projection_dim': PROJECTION_DIM,
                         'queue_size': QUEUE_SIZE, 'temperature': TEMPERATURE,
                         'momentum': MOCO_MOMENTUM,
                         'batch_size': BATCH_SIZE, 'lr': SSL_LR, 'epochs': SSL_EPOCHS,
                         'edge_dropout': JOINT_EDGE_DROP, 'feat_mask': JOINT_FEAT_MASK,
                         'node_drop': JOINT_NODE_DROP, 'subgraph': JOINT_SUBGRAPH,
                         'use_masked_recon': USE_MASKED_RECON,
                         'recon_weight': RECON_WEIGHT,
                         'labels_used_in_ssl': False}}
    with open(report_path, 'w') as f: json.dump(report, f, indent=2)

    del online, target, recon_head, opt, sched, scaler
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return report

print("joint MoCo trainer ready")

joint MoCo trainer ready


## Cell 16 — Run joint pretraining + per-target probes

Set `RUN_JOINT = False` if you only want the zero-shot protocol.

In [19]:
RUN_JOINT = False
FORCE_JOINT_RETRAIN = False

joint_bundle = build_joint_bundle(norm_protocol='global', tag='joint') if RUN_JOINT else None

if RUN_JOINT:
    print("\n#####  Joint BYOL  #####")
    try:
        byol_report = train_joint_byol(joint_bundle, force_retrain=FORCE_JOINT_RETRAIN)
    except Exception as e:
        import traceback; traceback.print_exc()
        byol_report = {'error': str(e)}
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    print("\n#####  Joint MoCo  #####")
    try:
        moco_report = train_joint_moco(joint_bundle, force_retrain=FORCE_JOINT_RETRAIN)
    except Exception as e:
        import traceback; traceback.print_exc()
        moco_report = {'error': str(e)}
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## Cell 17 — Evaluate joint encoders per dataset (frozen probe)

In [20]:
JOINT_PROBE_CSV = 'cross_dataset_joint_probe_results.csv'

if RUN_JOINT and os.path.isdir('ssl_checkpoints_joint'):
    joint_encoder_byol = 'ssl_checkpoints_joint/encoder_best.pth'
    joint_encoder_moco = 'ssl_checkpoints_joint_moco/encoder_best.pth'

    rows = []
    for ssl, ck_path in [('byol', joint_encoder_byol), ('moco', joint_encoder_moco)]:
        if not os.path.isfile(ck_path):
            print(f"[skip] joint {ssl} encoder missing at {ck_path}")
            continue
        enc, ckpt = load_encoder(ck_path)
        for target in DATASET_CONFIG:
            tgt = load_dataset_bundle(target, norm_stats=GLOBAL_NS,
                                      cache_key=f"{target}__global")
            if len(set(tgt['probe_train_bin'])) < 2:
                print(f"  [skip] {ssl}/{target}: single-class probe train")
                continue
            X_tr = embed_graphs(enc, tgt['probe_train_graphs'], DEVICE)
            X_va = embed_graphs(enc, tgt['val_graphs'],         DEVICE)
            X_te = embed_graphs(enc, tgt['test_graphs'],        DEVICE)
            r = frozen_linear_probe(X_tr, tgt['probe_train_bin'],
                                    X_va, tgt['val_bin'],
                                    X_te, tgt['test_bin'])
            rows.append({
                'source': 'joint', 'target': target, 'ssl': ssl, 'norm': 'global',
                'ROC': r['test_roc'], 'PR': r['test_pr'], 'F1': r['test_f1_binary'],
                'FPR': r['test_fpr'], 'FNR': r['test_fnr'],
                'threshold': r['threshold'],
                'val_ROC': r['val_roc'], 'val_mF1': r['val_macro_f1'],
                'knn_ROC': r['test_knn_roc'],
                'src_epoch': int(ckpt.get('epoch', -1)),
                'src_val_roc': float(ckpt.get('val_roc_auc', float('nan'))),
            })
            print(f"  joint/{ssl:<5} -> {target:<6}  "
                  f"ROC {r['test_roc']:.4f}  F1 {r['test_f1_binary']:.4f}")
        del enc
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    df_joint = pd.DataFrame(rows)
    df_joint.to_csv(JOINT_PROBE_CSV, index=False)
    print(f"\nSaved {JOINT_PROBE_CSV}  ({len(df_joint)} rows)")
else:
    df_joint = pd.DataFrame()
    print("Joint pretraining not run or no checkpoints.")

Joint pretraining not run or no checkpoints.


## Cell 18 — Protocol B · Few-shot fine-tune on target (encoder initialised from source)

This is the smallest-fidelity protocol here: it just measures the *frozen encoder + linear probe* at various target label budgets, which is exactly the PPT-GNN few-shot protocol. Full TFT fine-tuning lives in `3_1_phase_TFT.ipynb`; for cross-dataset we use the probe because it isolates the SSL representation.

In [21]:
FRACTIONS = [0.05, 0.25, 0.50, 1.00]
FEWSHOT_CSV = 'cross_dataset_fewshot_results.csv'

def few_shot_probe(enc, tgt_bundle, frac, seed=SEED):
    """Probe trained on `frac` of target probe_train labels."""
    X_tr = embed_graphs(enc, tgt_bundle['probe_train_graphs'], DEVICE)
    X_va = embed_graphs(enc, tgt_bundle['val_graphs'],         DEVICE)
    X_te = embed_graphs(enc, tgt_bundle['test_graphs'],        DEVICE)
    y_tr = tgt_bundle['probe_train_bin']

    rng = np.random.default_rng(seed)
    n_use = max(20, int(frac * len(X_tr)))
    # stratified subsample
    idx_pos = np.where(y_tr == 1)[0]; idx_neg = np.where(y_tr == 0)[0]
    n_pos = min(len(idx_pos), n_use // 2)
    n_neg = min(len(idx_neg), n_use - n_pos)
    sel = np.concatenate([rng.choice(idx_pos, n_pos, replace=False),
                          rng.choice(idx_neg, n_neg, replace=False)])
    rng.shuffle(sel)
    return frozen_linear_probe(
        X_tr[sel], y_tr[sel],
        X_va, tgt_bundle['val_bin'],
        X_te, tgt_bundle['test_bin']), int(n_use)


if os.path.isfile(FEWSHOT_CSV):
    df_fs = pd.read_csv(FEWSHOT_CSV)
    print(f"[cache] loaded {FEWSHOT_CSV}  ({len(df_fs)} rows)")
else:
    rows = []
    for ssl in SSL_METHODS:
        for src in DATASET_CONFIG:
            if not os.path.isfile(encoder_path_for(src, ssl)): continue
            enc, ckpt = load_encoder(encoder_path_for(src, ssl))
            for tgt in DATASET_CONFIG:
                tgt_bundle = load_dataset_bundle(
                    tgt,
                    norm_stats=(load_self_stats(src) if src != tgt else None),
                    cache_key=(f"{tgt}__src_{src}" if src != tgt else f"{tgt}__self"))
                if len(set(tgt_bundle['probe_train_bin'])) < 2: continue
                for frac in FRACTIONS:
                    r, n_use = few_shot_probe(enc, tgt_bundle, frac)
                    rows.append({
                        'ssl': ssl, 'source': src, 'target': tgt,
                        'frac': frac, 'n_labeled': n_use,
                        'ROC': r['test_roc'], 'F1': r['test_f1_binary'],
                        'PR':  r['test_pr'],
                        'FNR': r['test_fnr'], 'FPR': r['test_fpr'],
                    })
                    print(f"  {ssl:<5} {src:>6}->{tgt:<6} frac={frac:>4.2f}  "
                          f"n={n_use:>4}  ROC {r['test_roc']:.4f}  F1 {r['test_f1_binary']:.4f}")
            del enc
            gc.collect()
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    df_fs = pd.DataFrame(rows)
    df_fs.to_csv(FEWSHOT_CSV, index=False)
    print(f"\nSaved {FEWSHOT_CSV}  ({len(df_fs)} rows)")

  byol  gotham->gotham frac=0.05  n= 138  ROC 0.8158  F1 0.1695
  byol  gotham->gotham frac=0.25  n= 693  ROC 0.9141  F1 0.3265
  byol  gotham->gotham frac=0.50  n=1386  ROC 0.9562  F1 0.7385
  byol  gotham->gotham frac=1.00  n=2772  ROC 0.9736  F1 0.8874
  byol  gotham->nf_bot frac=0.05  n= 200  ROC 0.9374  F1 0.9480
  byol  gotham->nf_bot frac=0.25  n=1000  ROC 0.7063  F1 0.8855
  byol  gotham->nf_bot frac=0.50  n=2000  ROC 0.7837  F1 0.9185
  byol  gotham->nf_bot frac=1.00  n=4000  ROC 0.8153  F1 0.9478
  byol  gotham->nf_ton frac=0.05  n= 200  ROC 0.2683  F1 0.0957
  byol  gotham->nf_ton frac=0.25  n=1000  ROC 0.6546  F1 0.8952
  byol  gotham->nf_ton frac=0.50  n=2000  ROC 0.5487  F1 0.8901
  byol  gotham->nf_ton frac=1.00  n=4000  ROC 0.3611  F1 0.3772
  byol  nf_bot->gotham frac=0.05  n= 138  ROC 0.8040  F1 0.5913
  byol  nf_bot->gotham frac=0.25  n= 693  ROC 0.8689  F1 0.6667
  byol  nf_bot->gotham frac=0.50  n=1386  ROC 0.9544  F1 0.6748
  byol  nf_bot->gotham frac=1.00  n=2772

## Cell 19 — Summary JSON across all three protocols

In [22]:
summary = {
    'config': {
        'attack_threshold': CROSS_DATASET_ATTACK_THRESHOLD,
        'ssl_methods': SSL_METHODS,
        'datasets': list(DATASET_CONFIG.keys()),
        'ssl_epochs': SSL_EPOCHS,
        'fewshot_fractions': FRACTIONS,
        'seed': SEED,
    },
    'protocol_A_zero_shot':    df_probe.to_dict(orient='records') if not df_probe.empty else [],
    'protocol_A_transfer_gap': gap.to_dict(orient='records')     if not gap.empty     else [],
    'protocol_B_fewshot':      df_fs.to_dict(orient='records')   if not df_fs.empty    else [],
    'protocol_C_joint':        df_joint.to_dict(orient='records') if not df_joint.empty else [],
}
with open('cross_dataset_ssl_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("Saved cross_dataset_ssl_summary.json")

# ---- quick headline table ----
print("\n" + "="*100)
print("HEADLINE · zero-shot transfer ROC (source-normalised)")
print("="*100)
if not df_probe.empty:
    headline = df_probe[df_probe['norm'].isin(['source','self','global'])]
    pivot = headline.pivot_table(
        index=['ssl','source'], columns=['target','norm'],
        values='ROC', aggfunc='first')
    print(pivot.round(4).to_string())

print("\n" + "="*100)
print("HEADLINE · mean transfer gap (ROC)")
print("="*100)
if not gap.empty:
    print(gap.groupby(['ssl','norm'])['transfer_gap']
             .agg(['mean','std','count']).round(4).to_string())

Saved cross_dataset_ssl_summary.json

HEADLINE · zero-shot transfer ROC (source-normalised)
target       gotham                  nf_bot                  nf_ton                
norm         global    self  source  global    self  source  global    self  source
ssl  source                                                                        
byol gotham     NaN  0.9739     NaN  0.8918     NaN  0.7868  0.2856     NaN  0.3647
     nf_bot  0.9594     NaN  0.9601     NaN  0.8806     NaN  0.3750     NaN  0.3223
     nf_ton  0.9739     NaN  0.9821  0.7499     NaN  0.7420     NaN  0.6147     NaN
moco gotham     NaN  0.9700     NaN  0.9466     NaN  0.9001  0.2613     NaN  0.4299
     nf_bot  0.9735     NaN  0.9658     NaN  0.8229     NaN  0.6598     NaN  0.4999
     nf_ton  0.9600     NaN  0.9744  0.7003     NaN  0.7635     NaN  0.6551     NaN

HEADLINE · mean transfer gap (ROC)
               mean     std  count
ssl  norm                         
byol global  0.1171  0.1420      6
     source

## Cell 20 — Visualisations

In [23]:
def _save(fig, name):
    p = os.path.join(PLOTS_DIR, name)
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"[plot] {p}")

# --- 1. Heatmap: source × target ROC, one panel per (ssl, norm) ---
if not df_probe.empty:
    combos = sorted(df_probe[['ssl','norm']].drop_duplicates().itertuples(index=False))
    n = len(combos)
    fig, axes = plt.subplots(1, n, figsize=(4.2*n, 4.6), sharey=True)
    if n == 1: axes = [axes]
    for ax, (ssl, norm) in zip(axes, combos):
        sub = df_probe[(df_probe['ssl'] == ssl) & (df_probe['norm'] == norm)]
        if sub.empty: continue
        piv = sub.pivot_table(index='source', columns='target',
                               values='ROC', aggfunc='first')
        im = ax.imshow(piv.values, cmap='viridis', vmin=0.5, vmax=1.0)
        ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=20)
        ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                v = piv.values[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{v:.3f}", ha='center', va='center',
                            color='white' if v < 0.75 else 'black', fontsize=9)
        ax.set_title(f"{ssl.upper()} · {norm}")
        plt.colorbar(im, ax=ax, fraction=0.045)
    fig.suptitle('Zero-shot transfer ROC — source (rows) → target (columns)',
                 y=1.02, fontweight='bold')
    _save(fig, 'protoA_heatmap_ROC.png')

# --- 2. Transfer gap bars ---
if not gap.empty:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    gap_plot = gap.copy()
    gap_plot['pair'] = gap_plot['source'] + '→' + gap_plot['target']
    gap_plot = gap_plot.sort_values(['ssl','norm','pair'])
    xs = np.arange(len(gap_plot))
    cols = [('#1f77b4' if s == 'byol' else '#2ca02c') for s in gap_plot['ssl']]
    ax.bar(xs, gap_plot['transfer_gap'], color=cols, edgecolor='black', lw=0.4)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{r.pair}\n({r.ssl}/{r.norm})" for r in gap_plot.itertuples()],
                        rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('ROC gap (in-domain − transfer)')
    ax.set_title('Cross-dataset transfer gap — smaller is better (0 = perfect transfer)')
    ax.grid(alpha=0.3, axis='y')
    _save(fig, 'protoA_transfer_gap.png')

# --- 3. Few-shot curves ---
if not df_fs.empty:
    for ssl in SSL_METHODS:
        s = df_fs[df_fs['ssl'] == ssl]
        if s.empty: continue
        fig, ax = plt.subplots(figsize=(8, 4.8))
        for (src, tgt), g in s.groupby(['source','target']):
            g = g.sort_values('frac')
            label = f"{src}→{tgt}" if src != tgt else f"{src} (in-domain)"
            style = '-' if src != tgt else '--'
            ax.plot(g['frac']*100, g['ROC'], style, marker='o', lw=1.6, label=label)
        ax.set_xlabel('% labelled target graphs')
        ax.set_ylabel('Test ROC-AUC')
        ax.set_ylim(0.5, 1.02)
        ax.set_title(f'Few-shot transfer · {ssl.upper()} · frozen probe')
        ax.legend(fontsize=8, ncol=2)
        ax.grid(alpha=0.3)
        _save(fig, f'protoB_fewshot_{ssl}.png')

# --- 4. Joint vs single-dataset encoders ---
if not df_joint.empty and not df_probe.empty:
    fig, ax = plt.subplots(figsize=(9, 4.8))
    targets = sorted(DATASET_CONFIG.keys())
    x = np.arange(len(targets)); w = 0.24
    for i, (label, dfx, colour) in enumerate([
            ('self',   df_probe[df_probe['source'] == df_probe['target']],
             '#7f7f7f'),
            ('source', df_probe[(df_probe['source'] != df_probe['target']) &
                                (df_probe['norm'] == 'source')],
             '#1f77b4'),
            ('joint',  df_joint, '#ff7f0e')]):
        vals = []
        for t in targets:
            s = dfx[dfx['target'] == t]
            vals.append(s['ROC'].mean() if not s.empty else np.nan)
        ax.bar(x + (i-1)*w, vals, w, label=label, color=colour,
               edgecolor='black', lw=0.4)
    ax.set_xticks(x); ax.set_xticklabels(targets)
    ax.set_ylim(0.5, 1.02); ax.set_ylabel('Test ROC-AUC (mean)')
    ax.set_title('Self vs cross-source vs joint-pretrained encoder')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')
    _save(fig, 'protoC_joint_vs_single.png')

print(f"\nAll plots under {os.path.abspath(PLOTS_DIR)}/")

[plot] plots/cross_dataset\protoA_heatmap_ROC.png
[plot] plots/cross_dataset\protoA_transfer_gap.png
[plot] plots/cross_dataset\protoB_fewshot_byol.png
[plot] plots/cross_dataset\protoB_fewshot_moco.png

All plots under c:\Users\User\Desktop\MC-STGAT-IDS\MC-STGAT-IDS-V3\plots\cross_dataset/
